# 01. Модель надземної біомаси за GEDI L4A і Sentinel-1/2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yevheniikhan/kyiv-forest-agentic-ai/blob/main/notebooks/01_biomass_gedi_model.ipynb)

**Ділянка:** Боярська лісова дослідна станція та околиці (Київська область), близько 21 × 22 км.
**Еталон:** відбитки лідара GEDI L4A (AGBD, Мг/га; діаметр відбитка 25 м; 2019–2023).
**Предиктори:** медіанний композит Sentinel-2 (червень–серпень 2022), Sentinel-1 VV/VH, рельєф Copernicus DEM.
**Модель:** random forest і градієнтний бустинг.
**Валідація:** випадкова та просторова блокова крос-валідація (блоки 2 × 2 км, 5 груп).

Записник показує, як навчати й чесно перевіряти модель. Відбитки GEDI самі мають похибку моделі L4A,
тому отримані метрики описують узгодженість із GEDI, а не з наземною таксацією.
Координати пробних площ НІЛ не використовуються.

In [ ]:
# Код репозиторію і бібліотеки
import os, sys
if not os.path.exists('/content/kyiv-forest-agentic-ai'):
    !git clone -q https://github.com/yevheniikhan/kyiv-forest-agentic-ai /content/kyiv-forest-agentic-ai
sys.path.insert(0, '/content/kyiv-forest-agentic-ai/src')
!pip -q install geemap pyproj

import ee
PROJECT = 'gen-lang-client-0200925587'   # замініть на свій проєкт Google Cloud з увімкненим Earth Engine
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()                     # вікно авторизації підтверджує користувач
    ee.Initialize(project=PROJECT)
print('Earth Engine:', ee.String('готово').getInfo())

## 1. Вибірка відбитків GEDI з предикторами

In [ ]:
from kyivforest import config as C, ee_data as D, validate as V
import pandas as pd, numpy as np

fc = D.training_samples(max_n=4000)
rows = fc.map(lambda f: f.set({'lon': f.geometry().coordinates().get(0),
                               'lat': f.geometry().coordinates().get(1)})).getInfo()['features']
df = pd.DataFrame([r['properties'] for r in rows])
df = df[(df.agbd >= C.CHECKS['agbd_range'][0]) & (df.agbd <= C.CHECKS['agbd_range'][1])].dropna()
FEATURES = [c for c in df.columns if c not in ('agbd', 'lon', 'lat')]
df = V.add_blocks(df)
print(f'Відбитків: {len(df)}, просторових блоків: {df.block.nunique()}')
df.agbd.describe().round(1)

## 2. Випадкова і просторова крос-валідація

Випадкова CV перемішує сусідні відбитки між навчанням і перевіркою, тому завищує точність. Просторова CV тримає цілі блоки поза навчанням і ближча до умов перенесення моделі на нову територію.

In [ ]:
res = []
for kind in ('rf', 'gbm'):
    for scheme in ('random', 'spatial'):
        oof, m = V.cross_validate(df, FEATURES, kind=kind, scheme=scheme)
        res.append({'модель': kind, 'схема CV': scheme, **{k: round(v, 3) for k, v in m.items()}})
        if kind == 'rf' and scheme == 'spatial':
            df['pred_spatial'] = oof
metrics = pd.DataFrame(res)
metrics.to_csv('metrics_cv.csv', index=False)
metrics

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
ax[0].scatter(df.agbd, df.pred_spatial, s=4, alpha=.35, color='#14524A')
lim = [0, np.percentile(df.agbd, 99)]
ax[0].plot(lim, lim, color='#9a6b2f', lw=1); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('GEDI L4A AGBD, Мг/га'); ax[0].set_ylabel('Прогноз RF, Мг/га')
ax[0].set_title('Просторова CV')
imp = V.importance(df, FEATURES).head(10)[::-1]
ax[1].barh(imp.index, imp.values, color='#14524A'); ax[1].set_title('Важливість предикторів (RF)')
plt.tight_layout(); plt.savefig('fig_cv_importance.png', dpi=200); plt.show()

## 3. Карта біомаси і підсумки для ділянки

In [ ]:
import geemap
agb = D.ee_rf_map(fc, FEATURES)
st = D.area_stats(agb).getInfo()
forest_ha, agb_tot = st['ha_sum'], st['agb_tot_sum']
print(f'Площа лісу: {forest_ha:,.0f} га; середня AGB: {agb_tot/forest_ha:.1f} Мг/га; '
      f'надземна біомаса: {agb_tot/1e6:.2f} млн т')
vis = {'min': 0, 'max': 350, 'palette': ['f2efe6', 'c9d8c5', '8fb49a', '4f8a6f', '14524a', '0c3a34']}
m = geemap.Map(); m.centerObject(D.aoi(), 11)
m.addLayer(agb, vis, 'AGB, Мг/га'); m.add_colorbar(vis, label='AGB, Мг/га'); m

### Проєкція

Інтерактивна карта geemap показана в Web Mercator лише для перегляду. Площі обчислено через `ee.Image.pixelArea()` (фактична площа пікселя). Для друку і ГІС карту експортують у WGS 84 / UTM 36N (EPSG:32636), 30 м.

In [ ]:
from IPython.display import Image
print(D.export_map(agb).status()['state'], '– експорт GeoTIFF у UTM 36N запущено (вкладка Tasks / Google Drive)')
Image(url=D.preview_url(D.sentinel2().visualize(bands=['B4', 'B3', 'B2'], min=0, max=0.15)), width=480)

## 4. Що перевіряє дослідник

- чи не завеликий розрив між випадковою і просторовою CV (ознака перенавчання на локальну автокореляцію);
- чи немає систематичного зміщення на високих запасах (насичення оптичних індексів);
- чи узгоджується середнє значення з GEDI L4B і з регіональними даними (записник 02);
- для звітності потрібні наземні дані: GEDI є допоміжним еталоном.